# Partial Least Squares Regression: Theory and Practice

## A Textbook-Style Guide with Computational Exercises

**Course:** Machine Learning
**Total Homework Points:** 100 + 10 bonus

---

### How This Notebook Works

**Part 1** is a **textbook** — read it carefully. It develops the theory of PLS regression and compares it geometrically with Ridge regression. There are no problems to solve in Part 1; it is purely instructional.

**Parts 2–6** are **homework problems**. They ask you to implement algorithms and run experiments that bring the theory to life. Fill in the code cells marked `# YOUR CODE HERE` and the markdown cells marked *"Your answer here..."*.

### Prerequisites
You should be comfortable with:
- Linear regression and regularization (Ridge, Lasso)
- Bias-variance tradeoff and cross-validation
- Singular Value Decomposition (SVD) and PCA
- Basic matrix calculus ($\nabla_{\mathbf{w}} f(\mathbf{w})$, etc.)

### Submission
Submit this `.ipynb` with all code cells executed and all written answers filled in.

## Part 0: Setup and Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.cross_decomposition import PLSRegression
from sklearn.linear_model import Ridge, Lasso
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings("ignore")

from utils import (
    generate_low_rank_data,
    generate_sparse_data,
    generate_collinear_data,
    mse,
    relative_mse,
)

plt.rcParams.update({
    "figure.figsize": (10, 6),
    "font.size": 12,
    "axes.grid": True,
    "grid.alpha": 0.3,
})
sns.set_style("whitegrid")
print("Setup complete.")

---
# Part 1: The Theory of Partial Least Squares Regression

*This part is a textbook — read it carefully. There are no problems to solve here.*

## 1.1 What Problem Does PLS Solve?

Consider the standard regression setup: we observe a matrix $\mathbf{X} \in \mathbb{R}^{n \times p}$ of features and a response vector $\mathbf{y} \in \mathbb{R}^n$, and we want to predict $\mathbf{y}$ from $\mathbf{X}$.

When $p \gg n$ (many more features than samples), ordinary least squares (OLS) fails: the system $\mathbf{X}\boldsymbol{\beta} = \mathbf{y}$ is underdetermined, and $\mathbf{X}^T\mathbf{X}$ is singular. Even when $p < n$, if features are highly collinear, OLS estimates are wildly unstable.

Two classical remedies are:

| Method | Strategy | Hyperparameter |
|--------|----------|----------------|
| **Ridge** | Shrink all coefficients toward zero | $\lambda$ (penalty strength) |
| **PLS** | Project onto $k$ latent directions that maximize covariance with $\mathbf{y}$ | $k$ (number of components) |

Both are forms of **regularization**, but they regularize in fundamentally different ways. This chapter develops the theory of PLS and shows precisely when and why it differs from Ridge.

## 1.2 The PLS Objective: Maximizing Covariance

Assume $\mathbf{X}$ and $\mathbf{y}$ are centered (mean-subtracted). PLS finds successive **weight vectors** $\mathbf{w}_1, \mathbf{w}_2, \ldots$ that define latent directions in feature space.

### The first PLS component

The first weight vector solves:

$$\mathbf{w}_1 = \arg\max_{\|\mathbf{w}\| = 1} \; \text{Cov}^2(\mathbf{X}\mathbf{w},\; \mathbf{y})$$

Since $\mathbf{X}$ and $\mathbf{y}$ are centered, $\text{Cov}(\mathbf{X}\mathbf{w}, \mathbf{y}) = \frac{1}{n-1}\mathbf{w}^T\mathbf{X}^T\mathbf{y}$, so we are maximizing:

$$(\mathbf{w}^T \mathbf{X}^T \mathbf{y})^2 \quad \text{subject to} \quad \|\mathbf{w}\| = 1$$

By the **Cauchy-Schwarz inequality**, $(\mathbf{w}^T \mathbf{z})^2 \leq \|\mathbf{w}\|^2 \|\mathbf{z}\|^2$ with equality when $\mathbf{w} \propto \mathbf{z}$. Setting $\mathbf{z} = \mathbf{X}^T\mathbf{y}$:

$$\boxed{\mathbf{w}_1 = \frac{\mathbf{X}^T\mathbf{y}}{\|\mathbf{X}^T\mathbf{y}\|}}$$

This is beautifully simple: the first PLS direction is proportional to the **marginal regression** of $\mathbf{y}$ on each feature. Features that are individually correlated with $\mathbf{y}$ receive large weight.

### Contrast with PCA

PCA finds directions maximizing $\text{Var}(\mathbf{X}\mathbf{w}) = \mathbf{w}^T \mathbf{X}^T\mathbf{X} \mathbf{w}$, i.e., the **variance** of the projection, ignoring $\mathbf{y}$ entirely. The leading PCA direction is the top eigenvector of $\mathbf{X}^T\mathbf{X}$.

**When PCA fails as a regression preprocessing step:** If the direction of greatest variance in $\mathbf{X}$ is *orthogonal* to the signal relevant to $\mathbf{y}$, PCA regression with few components will discard the predictive information.

> **Example.** Suppose $\mathbf{X}$ has 100 features. Features 1–99 are driven by a high-variance nuisance factor (say, batch effects), while feature 100 alone predicts $\mathbf{y}$. PCA's first component captures the nuisance; PLS's first component captures feature 100 (because it has the highest covariance with $\mathbf{y}$).

PLS is therefore a form of **supervised dimensionality reduction**: it finds low-dimensional projections that are predictive, not just high-variance.

## 1.3 The NIPALS Algorithm

PLS extracts components sequentially using deflation. The standard algorithm for PLS1 (univariate $\mathbf{y}$) is called **NIPALS** (Nonlinear Iterative Partial Least Squares):

---

**Input:** Centered $\mathbf{X} \in \mathbb{R}^{n \times p}$, centered $\mathbf{y} \in \mathbb{R}^n$, number of components $k$.

**For** $a = 1, \ldots, k$:

1. **Weight vector:** $\;\mathbf{w}_a = \mathbf{X}^T \mathbf{y} \;/\; \|\mathbf{X}^T \mathbf{y}\|$

2. **Score vector:** $\;\mathbf{t}_a = \mathbf{X} \mathbf{w}_a$

3. **X-loading:** $\;\mathbf{p}_a = \mathbf{X}^T \mathbf{t}_a \;/\; (\mathbf{t}_a^T \mathbf{t}_a)$

4. **Y-loading:** $\;q_a = \mathbf{y}^T \mathbf{t}_a \;/\; (\mathbf{t}_a^T \mathbf{t}_a)$

5. **Deflate:** $\;\mathbf{X} \leftarrow \mathbf{X} - \mathbf{t}_a \mathbf{p}_a^T$, $\;\;\mathbf{y} \leftarrow \mathbf{y} - q_a \mathbf{t}_a$

---

**Why deflation?** After extracting component $a$, we remove its contribution from both $\mathbf{X}$ and $\mathbf{y}$ so that the next component captures *new* covariance structure. Without deflation, every component would be identical to the first.

### Recovery of regression coefficients

After extracting all $k$ components, collect:
- $\mathbf{W} = [\mathbf{w}_1, \ldots, \mathbf{w}_k] \in \mathbb{R}^{p \times k}$ (weights)
- $\mathbf{P} = [\mathbf{p}_1, \ldots, \mathbf{p}_k] \in \mathbb{R}^{p \times k}$ (X-loadings)
- $\mathbf{q} = [q_1, \ldots, q_k]^T \in \mathbb{R}^k$ (Y-loadings)

The regression coefficient vector is:

$$\boxed{\hat{\boldsymbol{\beta}}_{\text{PLS}} = \mathbf{W}(\mathbf{P}^T\mathbf{W})^{-1}\mathbf{q}}$$

The matrix $\mathbf{R} = \mathbf{W}(\mathbf{P}^T\mathbf{W})^{-1}$ is sometimes called the **rotation matrix**; the columns of $\mathbf{X}\mathbf{R}$ give orthogonal score vectors.

### Prediction on new data

For centered training data with means $\bar{\mathbf{x}}$ and $\bar{y}$:

$$\hat{y}_{\text{new}} = (\mathbf{x}_{\text{new}} - \bar{\mathbf{x}})^T \hat{\boldsymbol{\beta}}_{\text{PLS}} + \bar{y}$$

## 1.4 How PLS and Ridge Regularize: A Geometric Comparison

Both PLS and Ridge can be understood through the **Singular Value Decomposition** of $\mathbf{X}$. Write $\mathbf{X} = \mathbf{U}\boldsymbol{\Sigma}\mathbf{V}^T$, where $\boldsymbol{\Sigma} = \text{diag}(\sigma_1, \ldots, \sigma_r)$ with $r = \text{rank}(\mathbf{X})$.

### Ridge filter factors

The Ridge solution is:
$$\hat{\boldsymbol{\beta}}_{\text{ridge}} = \mathbf{V} \text{diag}\!\left(\frac{\sigma_j}{\sigma_j^2 + \lambda}\right) \mathbf{U}^T \mathbf{y}$$

This applies a **filter factor** $f_j^{\text{ridge}} = \frac{\sigma_j^2}{\sigma_j^2 + \lambda}$ to each SVD component. Key properties:
- $f_j \in [0, 1]$ for all $j$
- $f_j$ is **monotonically increasing** in $\sigma_j$: large-singular-value directions are preserved; small ones are shrunk
- Ridge treats all directions symmetrically in the SVD basis — it never *selects* directions based on their relationship with $\mathbf{y}$

**Geometric picture:** Ridge "uniformly dampens" low-variance directions, regardless of whether they carry signal about $\mathbf{y}$.

### PLS filter factors

PLS also applies filter factors in the SVD basis, but they are **not monotone** in the singular values. A PLS direction is retained if it has high covariance with $\mathbf{y}$, even if its singular value is small. Conversely, a high-variance direction that is orthogonal to $\mathbf{y}$ may receive a filter factor near zero.

| Property | Ridge | PLS ($k$ components) |
|----------|-------|---------------------|
| Filter factors monotone in $\sigma_j$ | Yes | **No** |
| Uses $\mathbf{y}$ information | No (only $\mathbf{X}$) | **Yes** |
| Effective dimensionality | Continuous (via $\lambda$) | Discrete (integer $k$) |
| Unregularized limit | $\lambda \to 0$ gives OLS | $k = \min(n, p)$ gives OLS |
| Computational cost | $O(p^2 n)$ (or $O(n^2 p)$) | $O(knp)$ |

### When does this matter?

Consider a data-generating process where $\mathbf{y}$ depends on $\mathbf{X}$ through $k_{\text{true}} \ll p$ latent factors:

$$\mathbf{X} = \mathbf{T}\mathbf{P}^T + \mathbf{E}, \qquad \mathbf{y} = \mathbf{T}\boldsymbol{\gamma} + \epsilon$$

Here $\mathbf{T} \in \mathbb{R}^{n \times k_{\text{true}}}$ are latent scores, $\mathbf{P}$ maps them to observed features, and $\boldsymbol{\gamma}$ gives the latent regression weights.

- **PLS** with $k = k_{\text{true}}$ components recovers the signal space almost exactly. It achieves low bias (captures the true subspace) and low variance (only $k_{\text{true}}$ parameters to estimate).
- **Ridge** must estimate all $p$ coefficients, merely shrinking them. It pays a variance cost for the $p - k_{\text{true}}$ noise directions, even though they carry no signal.

**Bottom line:** PLS has a strong *inductive bias* that the signal is low-rank. When this matches reality, PLS wins. When the true coefficient vector is sparse (many zeros, no latent structure), PLS's whole-subspace approach isn't as effective as Ridge or Lasso.

## 1.5 The $p \gg n$ Regime

When $p > n$, the matrix $\mathbf{X}^T\mathbf{X} \in \mathbb{R}^{p \times p}$ has rank at most $n < p$, so it is **singular**. OLS requires inverting $\mathbf{X}^T\mathbf{X}$, which is impossible.

**How Ridge resolves this.** Ridge replaces $\mathbf{X}^T\mathbf{X}$ with $\mathbf{X}^T\mathbf{X} + \lambda\mathbf{I}$. Since $\lambda > 0$, all eigenvalues are shifted above zero: if $\sigma_j^2$ was an eigenvalue of $\mathbf{X}^T\mathbf{X}$, the new eigenvalue is $\sigma_j^2 + \lambda > 0$. The matrix is always invertible.

**How PLS resolves this.** PLS never inverts $\mathbf{X}^T\mathbf{X}$ at all. Each NIPALS step only uses matrix-vector products $\mathbf{X}^T\mathbf{v}$ and $\mathbf{X}\mathbf{w}$, which are well-defined regardless of the rank of $\mathbf{X}$. The implicit regularization comes from truncating at $k < \text{rank}(\mathbf{X})$ components.

This makes PLS especially natural for spectroscopy, genomics, neuroimaging, and other domains where $p$ can be in the thousands or millions while $n$ is in the tens or hundreds.

## 1.6 Summary: When to Use PLS vs. Ridge

| Scenario | Recommended Method | Why |
|----------|-------------------|-----|
| Low-rank latent structure, $p \gg n$ | **PLS** | PLS's inductive bias matches the data; reduces effective dimension to $k$ |
| Sparse true coefficients | **Lasso** (or Ridge) | PLS doesn't perform variable selection; Lasso's $\ell_1$ penalty does |
| Collinear feature groups | **PLS** (often) | Collinear groups create latent factors; PLS extracts them naturally |
| Abundant data ($n \gg p$) | **Ridge ≈ PLS** | Both converge to OLS; regularization matters less |
| No prior knowledge of structure | **Try both + CV** | Let cross-validation decide |

With this theoretical foundation, you're ready for the computational exercises in Parts 2–6.

---
# Part 2: Implement PLS via NIPALS (20 pts)

*From here on, everything is homework. Fill in the code and written answers.*

### Problem 2.1 — NIPALS Implementation (12 pts)

Using the algorithm described in Section 1.3, implement NIPALS for PLS1 from scratch.

**Requirements:**
- Center $\mathbf{X}$ and $\mathbf{y}$ internally
- Return all components needed for prediction on new data
- Handle the intercept correctly

In [ ]:
def nipals_pls1(X, y, n_components):
    """
    PLS1 regression via the NIPALS algorithm.
    
    Parameters
    ----------
    X : ndarray of shape (n_samples, n_features)
    y : ndarray of shape (n_samples,)
    n_components : int
    
    Returns
    -------
    W : ndarray of shape (n_features, n_components) — weight vectors
    T : ndarray of shape (n_samples, n_components)  — scores
    P : ndarray of shape (n_features, n_components) — X-loadings
    q_vec : ndarray of shape (n_components,)         — Y-loadings
    beta : ndarray of shape (n_features,)            — regression coefficients
    x_mean : ndarray of shape (n_features,)
    y_mean : float
    """
    n, p = X.shape
    
    # Center
    x_mean = X.mean(axis=0)
    y_mean = y.mean()
    Xc = X - x_mean
    yc = y - y_mean
    
    # Allocate
    W = np.zeros((p, n_components))
    T = np.zeros((n, n_components))
    P = np.zeros((p, n_components))
    q_vec = np.zeros(n_components)
    
    # YOUR CODE HERE
    for a in range(n_components):
        # Step 1: w_a = X^T y / ||X^T y||
        pass
        
        # Step 2: t_a = X w_a
        pass
        
        # Step 3: p_a = X^T t_a / (t_a^T t_a)
        pass
        
        # Step 4: q_a = y^T t_a / (t_a^T t_a)
        pass
        
        # Step 5: Deflate X and y
        pass
    
    # Compute regression coefficients: beta = W (P^T W)^{-1} q
    beta = np.zeros(p)  # YOUR CODE HERE
    
    return W, T, P, q_vec, beta, x_mean, y_mean


def predict_pls(X_new, beta, x_mean, y_mean):
    """Predict using fitted PLS model."""
    return (X_new - x_mean) @ beta + y_mean

### Problem 2.2 — Validation Against scikit-learn (8 pts)

Compare your implementation with `sklearn.cross_decomposition.PLSRegression`. They should agree to within numerical precision ($\sim 10^{-10}$).

In [ ]:
# Problem 2.2: Validate implementation
X_val, y_val, _ = generate_low_rank_data(n_samples=100, n_features=20, n_latent=3, seed=123)
X_test_val, y_test_val, _ = generate_low_rank_data(n_samples=200, n_features=20, n_latent=3, seed=456)
n_comp = 3

# YOUR CODE HERE
# 1. Fit your nipals_pls1, get predictions
# 2. Fit sklearn PLSRegression(n_components=3, scale=False), get predictions
# 3. Print max absolute difference in predictions and coefficients

---
# Part 3: PLS vs Ridge in the $p \gg n$ Regime (25 pts)

The theory in Section 1.4 predicts that PLS should outperform Ridge when the data has low-rank latent structure and samples are scarce. Let's test this.

### Problem 3.1 — Varying the True Latent Rank (15 pts)

**Setup:**
- `generate_low_rank_data`: $n = 50$ train, $p = 200$, $n_{\text{test}} = 500$
- Vary `n_latent` $\in \{2, 5, 10, 20\}$
- CV-tune PLS (`n_components` from 1–30) and Ridge ($\alpha$ from `np.logspace(-3, 3, 50)`)
- 20 random seeds; report mean $\pm$ standard error

**Produce two plots:**
1. Test MSE vs. `n_latent`
2. Relative coefficient MSE vs. `n_latent`

**Discuss:** When the latent rank is small, which method wins? What happens as it grows?

In [ ]:
# Problem 3.1: PLS vs Ridge — Varying latent rank
# ============================================================
n_train = 50
n_test = 500
n_features = 200
latent_ranks = [2, 5, 10, 20]
n_seeds = 20

# YOUR CODE HERE
# For each latent rank and seed:
#   1. Generate train/test data
#   2. CV-tune PLS and Ridge
#   3. Record test MSE and relative coefficient MSE

pass

In [ ]:
# Problem 3.1: Plotting
# YOUR CODE HERE — two error-bar plots side by side
pass

**Your Discussion (3.1):** Interpret the plots. When the true latent rank is small, which method wins? What happens as `n_latent` grows toward $n$?

*Your answer here...*

### Problem 3.2 — Learning Curves: Varying Sample Size (10 pts)

Fix `n_latent = 5`, $p = 200$. Vary $n \in \{20, 30, 50, 75, 100, 200\}$.

**Plot:** Test MSE vs. $n$ for both methods.

**Discuss:** At what sample size does Ridge start matching PLS? Why (bias-variance)?

In [ ]:
# Problem 3.2: Learning curves
sample_sizes = [20, 30, 50, 75, 100, 200]
n_latent_fixed = 5

# YOUR CODE HERE
pass

In [ ]:
# Problem 3.2: Plotting
# YOUR CODE HERE
pass

**Your Discussion (3.2):**

1. At what sample size does Ridge start matching PLS?
2. Explain in terms of bias-variance tradeoff.

*Your answer here...*

---
# Part 4: PLS vs Ridge Under Sparse Signals (25 pts)

Part 3 confirmed PLS's advantage on low-rank data. Now we flip the script: the true signal is **sparse** (few non-zero coefficients), with no latent structure.

### Problem 4.1 — Sparse Coefficients (15 pts)

**Setup:**
- `generate_sparse_data`: $n = 60$, $p = 150$
- Vary `n_informative` $\in \{3, 10, 30, 60\}$
- Two correlation settings: $\rho = 0.0$ and $\rho = 0.7$
- Include **Lasso** as a reference (sparse recovery is its strength)
- 20 seeds, test set of 500

**Create a 2×2 figure:**
- Rows: correlation setting
- Left column: grouped bar chart of test MSE (PLS / Ridge / Lasso)
- Right column: stem plots of true vs estimated coefficients (for `n_informative=10`, seed=0)

**Discuss:** Which method wins when the signal is sparse? How does correlation change things?

In [ ]:
# Problem 4.1: Sparse signals
n_train_4 = 60
n_features_4 = 150
n_test_4 = 500
informative_counts = [3, 10, 30, 60]
correlations = [0.0, 0.7]
n_seeds_4 = 20

# YOUR CODE HERE
pass

In [ ]:
# Problem 4.1: 2x2 figure
# YOUR CODE HERE
pass

**Your Discussion (4.1):**

1. When the true model is sparse, which method wins? Why?
2. How does feature correlation change the relative performance?
3. Under what conditions does PLS perform comparably to Ridge?

*Your answer here...*

### Problem 4.2 — Collinear Groups (10 pts)

Features in groups of collinear variables: a setting *between* low-rank and sparse.

**Setup:** `generate_collinear_data`: $n = 60$, $p = 100$, 5 groups of 10 + 50 noise features.

**Produce:** Bar chart of test MSE + stem plots showing coefficient recovery. Highlight group boundaries.

**Discuss:** Which method better recovers the group structure?

In [ ]:
# Problem 4.2: Collinear groups
n_train_42 = 60
n_features_42 = 100
n_groups = 5
group_size = 10

# YOUR CODE HERE
pass

In [ ]:
# Problem 4.2: Visualization
# YOUR CODE HERE — bar chart + coefficient stem plots with group boundaries
pass

**Your Discussion (4.2):** Which method better recovers the group structure? Why?

*Your answer here...*

---
# Part 5: Hyperparameter Sensitivity and Component Visualization (15 pts)

### Problem 5.1 — PLS Components vs Ridge $\alpha$: A CV Study (10 pts)

For a fixed setting ($n = 50$, $p = 100$, `n_latent = 3`, seed = 42):
1. Sweep PLS components 1–20: compute 5-fold CV MSE and test MSE
2. Sweep Ridge $\alpha$ over `np.logspace(-3, 5, 80)`: same
3. Plot side-by-side, marking CV-optimal points

**Discuss:** Which method's optimum is sharper? Which is more robust to hyperparameter mis-specification?

In [ ]:
# Problem 5.1: Hyperparameter sensitivity
n_train_5 = 50
n_features_5 = 100
n_latent_5 = 3
seed_5 = 42

X_train_5, y_train_5, beta_true_5 = generate_low_rank_data(n_train_5, n_features_5, n_latent_5, seed=seed_5)
X_test_5, y_test_5, _ = generate_low_rank_data(500, n_features_5, n_latent_5, seed=seed_5 + 1000)

pls_comp_range_5 = np.arange(1, 21)
ridge_alpha_range_5 = np.logspace(-3, 5, 80)

# YOUR CODE HERE
pass

In [ ]:
# Problem 5.1: Side-by-side plots
# YOUR CODE HERE
pass

**Your Discussion (5.1):** Compare the sharpness of each optimum. Which is more robust to hyperparameter mis-specification?

*Your answer here...*

### Problem 5.2 — PLS vs PCA Latent Components (5 pts)

Using the same dataset:
1. Extract 3 PLS scores and 3 PCA scores
2. Scatter-plot grid (2×3), color-coded by $\mathbf{y}$
3. Report correlation of each component with $\mathbf{y}$

**Discuss:** Which components are more aligned with the response?

In [ ]:
# Problem 5.2: PLS vs PCA components
# YOUR CODE HERE
pass

**Your Discussion (5.2):** Which components show a clearer relationship with $\mathbf{y}$? Why?

*Your answer here...*

---
# Part 6: Synthesis (10 pts)

### Problem 6.1 — Summary Table and Heatmap (5 pts)

Compile results from Parts 3–4 into:
1. A summary table with test MSE for PLS and Ridge across all conditions
2. A heatmap of PLS MSE / Ridge MSE ratios (< 1 = PLS wins)

In [ ]:
# Problem 6.1: Summary table and heatmap
# YOUR CODE HERE
pass

### Problem 6.2 — Written Analysis (5 pts)

Write a concise analysis addressing:
1. Under what conditions does PLS significantly outperform Ridge?
2. Under what conditions does Ridge match or beat PLS?
3. Bias-variance perspective on each method's regularization strategy.
4. Practical recommendation for a new dataset ($n = 80$, $p = 300$, unknown structure).

**Your Analysis (6.2):**

*Write your analysis here (1 page max)...*

---
# Bonus: The PLS Regularization Path (10 pts extra credit)

Plot coefficient paths for PLS (varying `n_components` from 1 to 30) and Ridge (varying $\alpha$ from high to low). Highlight the 5 features with largest true $|\beta_j|$.

**Discuss:** How do the paths differ? Does PLS show feature-selection-like behavior? Is it monotone?

In [ ]:
# Bonus: Coefficient paths
# YOUR CODE HERE
pass

**Your Discussion (Bonus):** How do the coefficient paths differ? Does PLS show feature-selection-like behavior? Is the PLS path monotone?

*Your answer here...*

*End of homework.*